In [3]:
from scapy.all import sniff, ARP
import datetime

arp_table = {}  

def log_event(event_type, src_ip, src_mac, anomaly="None"):
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log_entry = f"{timestamp}, {event_type}, {src_ip}, {src_mac}, {anomaly}\n"
    with open("arp_log.csv", "a") as logfile:
        logfile.write(log_entry)

def detect_arp_spoof(packet):
    if packet.haslayer(ARP) and packet[ARP].op == 2:  
        src_ip = packet[ARP].psrc
        src_mac = packet[ARP].hwsrc
        if src_ip in arp_table:
            if arp_table[src_ip] != src_mac:

                log_event("ARP_Reply", src_ip, src_mac, "Duplicate MAC Detected")
        else:
            arp_table[src_ip] = src_mac
            log_event("ARP_Reply", src_ip, src_mac)

sniff(store=False, prn=detect_arp_spoof, filter="arp")

<Sniffed: TCP:0 UDP:0 ICMP:0 Other:0>

In [ ]:
import logging
import random
import threading
import datetime
from scapy.all import sniff, ARP, conf

class ARPMonitor:
    def __init__(self, interface=None):
        self.arp_table = {}
        self.log_file = f"arp_log_{random.randint(1000,9999)}.csv"
        self.interface = interface or conf.iface
        self._setup_logging()

    def _setup_logging(self):
        logging.basicConfig(
            filename=self.log_file, 
            level=logging.INFO,
            format='%(asctime)s,%(message)s',
            datefmt='%Y-%m-%d %H:%M:%S'
        )

    def _log_event(self, event_type, src_ip, src_mac, anomaly="None"):
        log_message = f"{event_type},{src_ip},{src_mac},{anomaly}"
        logging.info(log_message)

    def detect_arp_anomalies(self, packet):
        if packet.haslayer(ARP) and packet[ARP].op == 2:  
            src_ip = packet[ARP].psrc
            src_mac = packet[ARP].hwsrc

            if src_ip in self.arp_table:
                if self.arp_table[src_ip] != src_mac:
                    self._log_event(
                        "ARP_ANOMALY", 
                        src_ip, 
                        src_mac, 
                        "MAC_CHANGE"
                    )
            else:
                self.arp_table[src_ip] = src_mac
                self._log_event("ARP_REPLY", src_ip, src_mac)

    def start_monitoring(self):
        monitoring_thread = threading.Thread(
            target=self._run_sniff, 
            daemon=True
        )
        monitoring_thread.start()

    def _run_sniff(self):
        sniff(
            iface=self.interface, 
            store=False, 
            prn=self.detect_arp_anomalies, 
            filter="arp"
        )

def main():
    arp_monitor = ARPMonitor()
    arp_monitor.start_monitoring()
    
    try:
        while True:
            pass
    except KeyboardInterrupt:
        print("\nARP monitoring stopped.")

if __name__ == "__main__":
    main()